# Semaine 12 – Évaluation pratique – Partie 1

## Objectifs de la session

À l'issue de cette évaluation pratique (1,5 heure), l'étudiant sera capable de :

- **Générer** et **analyser** des signaux numériques en Python.
- **Appliquer** la théorie de l'échantillonnage et **reconstruire** un signal.
- **Implémenter** la FFT et **interpréter** un spectre d'amplitude et de phase.
- **Concevoir** et **appliquer** des filtres numériques RIF et RII.
- **Analyser** l'effet du fenêtrage et du filtrage sur des signaux réels ou synthétiques.

## Format de l'évaluation

- **Durée** : 1 h 30
- **Modalité** : en laboratoire, sur machine, avec Python (Jupyter Notebook)
- **Documents autorisés** : notes de cours, notebooks des semaines précédentes
- **Barème indicatif** :
  - Partie A – Génération et analyse de signaux : **4 pts**
  - Partie B – Échantillonnage et reconstruction : **5 pts**
  - Partie C – FFT et analyse spectrale : **6 pts**
  - Partie D – Conception et application de filtres RIF/RII : **5 pts**
  - **Total : 20 points**

## Consignes

- Répondez à **toutes** les parties.
- **Commentez** chaque résultat obtenu (interprétation physique).
- Les graphiques doivent être **lisibles** (titre, axes, légende, grille).
- **Sauvegardez** régulièrement votre notebook.
- Le corrigé sera fourni à la fin de la séance.

# 0. Glossaire des acronymes utilisés

| Acronyme | Signification (anglais) | Signification (français) |
|---|---|---|
| **TNS / DSP** | Digital Signal Processing | Traitement Numérique du Signal |
| **FFT** | Fast Fourier Transform | Transformée de Fourier Rapide |
| **DFT** | Discrete Fourier Transform | Transformée de Fourier Discrète |
| **DTFT** | Discrete-Time Fourier Transform | Transformée de Fourier à Temps Discret |
| **RIF / FIR** | Finite Impulse Response | Réponse Impulsionnelle Finie |
| **RII / IIR** | Infinite Impulse Response | Réponse Impulsionnelle Infinie |
| **SNR** | Signal-to-Noise Ratio | Rapport Signal / Bruit |
| **CAN / ADC** | Analog-to-Digital Converter | Convertisseur Analogique-Numérique |
| **CNA / DAC** | Digital-to-Analog Converter | Convertisseur Numérique-Analogique |
| **Hz** | Hertz | Hertz (cycles par seconde) |
| **rad** | Radian | Radian (1 tour = 2π rad) |

> **Conventions** :
> - $x[n]$ : signal à temps discret
> - $X(e^{j\omega})$ : DTFT de $x[n]$
> - $X[k]$ : DFT de longueur $N$
> - $f_s$ : fréquence d'échantillonnage (Hz)
> - $h[n]$ : réponse impulsionnelle d'un filtre
> - $y[n]$ : sortie filtrée

In [1]:
# Imports communs à toutes les parties
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, ifft, fftfreq
from scipy.signal import find_peaks, freqz, butter, cheby1, filtfilt, lfilter

# Configuration des graphiques
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Reproductibilité
np.random.seed(42)

print("Bibliothèques chargées avec succès.")

Bibliothèques chargées avec succès.


# Partie A – Génération et analyse de signaux (4 points)

## A.1 – Synthèse d'un signal composé

**Question A.1 (1 pt)** : Générez un signal $x(t)$ composé de trois sinusoïdes :
- $f_1 = 50$ Hz, amplitude $1.0$
- $f_2 = 120$ Hz, amplitude $0.6$
- $f_3 = 250$ Hz, amplitude $0.3$

Durée : **0.5 seconde**, fréquence d'échantillonnage : $f_s = 2000$ Hz.

Tracez le signal temporel sur les 50 premières ms.

In [ ]:
# --- Votre code ici ---
fs = 2000
duration = 0.5
t = np.linspace(0, duration, int(fs*duration), endpoint=False)

f1, f2, f3 = 50, 120, 250
A1, A2, A3 = 1.0, 0.6, 0.3

x = A1*np.sin(2*np.pi*f1*t) + A2*np.sin(2*np.pi*f2*t) + A3*np.sin(2*np.pi*f3*t)

plt.figure(figsize=(12, 4))
plt.plot(t[:100], x[:100])
plt.xlabel('Temps (s)')
plt.ylabel('Amplitude')
plt.title('Signal composé (3 sinusoïdes)')
plt.tight_layout()
plt.show()

print(f"Nombre d'échantillons : {len(x)}")
print(f"Durée : {duration} s")
print(f"Fréquence d'échantillonnage : {fs} Hz")

### Interprétation A.1

- Le signal présente une **structure périodique complexe** due à la superposition de trois sinusoïdes.
- La période apparente la plus longue correspond à la **fondamentale** (50 Hz → 20 ms).
- Les oscillations rapides proviennent de la composante à 250 Hz.
- L'amplitude maximale théorique est $A_1 + A_2 + A_3 = 1.9$ (atteinte lorsque les trois sinusoïdes sont en phase).

## A.2 – Ajout de bruit

**Question A.2 (1 pt)** : Ajoutez un bruit gaussien blanc de variance $\sigma^2 = 0.05$ au signal $x(t)$.

1. Calculez le SNR (rapport signal/bruit) en dB.
2. Tracez le signal bruité sur les 50 premières ms.

In [ ]:
# --- Votre code ici ---
sigma = np.sqrt(0.05)
bruit = sigma * np.random.randn(len(x))
x_bruite = x + bruit

# Calcul du SNR
P_signal = np.mean(x**2)
P_bruit = np.mean(bruit**2)
SNR_dB = 10 * np.log10(P_signal / P_bruit)

plt.figure(figsize=(12, 4))
plt.plot(t[:100], x[:100], label='Signal original', alpha=0.7)
plt.plot(t[:100], x_bruite[:100], label='Signal bruité', alpha=0.5)
plt.xlabel('Temps (s)')
plt.ylabel('Amplitude')
plt.title('Signal original vs signal bruité')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Puissance signal : {P_signal:.4f}")
print(f"Puissance bruit  : {P_bruit:.4f}")
print(f"SNR : {SNR_dB:.2f} dB")

## A.3 – Analyse statistique

**Question A.3 (2 pts)** : Calculez et affichez :
1. La valeur moyenne du signal bruité.
2. L'écart-type du signal bruité.
3. L'énergie totale du signal bruité.

**Interprétation** : que peut-on dire du bruit ajouté ?

In [ ]:
# --- Votre code ici ---
moyenne = np.mean(x_bruite)
ecart_type = np.std(x_bruite)
energie = np.sum(x_bruite**2)

print(f"Moyenne          : {moyenne:.6f}")
print(f"Écart-type       : {ecart_type:.4f}")
print(f"Énergie totale   : {energie:.4f}")

### Interprétation A.3

- La **moyenne** est proche de 0 (bruit centré + sinusoïdes centrées).
- L'**écart-type** reflète la puissance totale du signal bruité.
- L'**énergie** augmente avec le bruit ajouté.
- Le bruit gaussien blanc a une densité spectrale **constante** sur toutes les fréquences → il affectera toutes les bandes du spectre de manière uniforme.

# Partie B – Échantillonnage et reconstruction (5 points)

## B.1 – Théorème de Nyquist

**Question B.1 (1 pt)** : Pour le signal de la partie A (fréquence max = 250 Hz) :
1. Quelle est la fréquence de Nyquist minimale nécessaire ?
2. La fréquence $f_s = 2000$ Hz respecte-t-elle ce critère ? Justifiez.
3. Que se passerait-il si on échantillonnait à 400 Hz ?

### Réponse B.1

**1.** La fréquence maximale du signal est $f_{\max} = 250$ Hz. Le théorème de Nyquist-Shannon impose :
$$f_s > 2 f_{\max} = 500 \text{ Hz}$$

**2.** Oui, $f_s = 2000$ Hz > 500 Hz. Le critère est largement respecté (facteur 4 de sur-échantillonnage).

**3.** Si $f_s = 400$ Hz < 500 Hz, il y aurait **repliement de spectre** (aliasing). La composante à 250 Hz se replierait à :
$$f_{\text{apparente}} = |250 - 400| = 150 \text{ Hz}$$
On observerait un pic parasite à 150 Hz au lieu de 250 Hz.

## B.2 – Simulation du repliement

**Question B.2 (2 pts)** : Échantillonnez le signal $x(t)$ à $f_s = 400$ Hz et comparez son spectre avec celui obtenu à $f_s = 2000$ Hz.

1. Générez les deux signaux échantillonnés.
2. Calculez leurs FFT respectives.
3. Identifiez les pics dans chaque cas.

In [ ]:
# --- Votre code ici ---
def generer_signal(fs, duration=0.5):
    t = np.linspace(0, duration, int(fs*duration), endpoint=False)
    x = A1*np.sin(2*np.pi*f1*t) + A2*np.sin(2*np.pi*f2*t) + A3*np.sin(2*np.pi*f3*t)
    return t, x

# Cas 1 : fs = 2000 Hz (correct)
fs1 = 2000
t1, x1 = generer_signal(fs1)
X1 = fft(x1)
freqs1 = fftfreq(len(x1), 1/fs1)[:len(x1)//2]
mag1 = 2*np.abs(X1[:len(x1)//2]) / len(x1)

# Cas 2 : fs = 400 Hz (sous-échantillonnage)
fs2 = 400
t2, x2 = generer_signal(fs2)
X2 = fft(x2)
freqs2 = fftfreq(len(x2), 1/fs2)[:len(x2)//2]
mag2 = 2*np.abs(X2[:len(x2)//2]) / len(x2)

# Tracé
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(freqs1, mag1)
axes[0].set_xlim(0, 500)
axes[0].set_title(f'Spectre correct (fs = {fs1} Hz)')
axes[0].set_xlabel('Fréquence (Hz)')
axes[0].set_ylabel('Magnitude')

axes[1].plot(freqs2, mag2, color='C1')
axes[1].set_xlim(0, 200)
axes[1].set_title(f'Spectre replié (fs = {fs2} Hz)')
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Magnitude')

plt.tight_layout()
plt.show()

# Détection automatique des pics
peaks1, _ = find_peaks(mag1, height=0.1)
peaks2, _ = find_peaks(mag2, height=0.1)
print(f"Pics détectés (fs=2000 Hz) : {freqs1[peaks1].round(1)} Hz")
print(f"Pics détectés (fs=400 Hz)  : {freqs2[peaks2].round(1)} Hz")

### Interprétation B.2

- Avec $f_s = 2000$ Hz, on retrouve exactement les trois pics attendus : **50, 120 et 250 Hz**.
- Avec $f_s = 400$ Hz, la composante à 250 Hz est **repliée** à $|250 - 400| = 150$ Hz. On observe donc des pics à **50, 120 et 150 Hz** : c'est le **repliement de spectre**.
- Ce phénomène est **irréversible** : une fois replié, il est impossible de distinguer la vraie fréquence de la fréquence apparente (sauf information a priori).
- **Solution pratique** : utiliser un **filtre anti-repliement** (passe-bas) avant l'échantillonnage.

## B.3 – Reconstruction

**Question B.3 (2 pts)** : Effectuez une reconstruction idéale du signal échantillonné à $f_s = 400$ Hz en utilisant la formule d'interpolation de Shannon :

$$x_r(t) = \sum_{n} x[n] \, \text{sinc}\left(\frac{t - nT_s}{T_s}\right)$$

Comparez avec le signal original. Concluez.

In [ ]:
# --- Votre code ici ---
def reconstruction_shannon(x_ech, fs, t_recons):
    """Reconstruction par interpolation sinc."""
    Ts = 1/fs
    n = np.arange(len(x_ech))
    xr = np.zeros_like(t_recons)
    for i, ni in enumerate(n):
        xr += x_ech[i] * np.sinc((t_recons - ni*Ts) / Ts)
    return xr

# Signal de référence haute résolution
t_ref = np.linspace(0, 0.05, 2000, endpoint=False)
x_ref = A1*np.sin(2*np.pi*f1*t_ref) + A2*np.sin(2*np.pi*f2*t_ref) + A3*np.sin(2*np.pi*f3*t_ref)

# Reconstruction à partir de fs = 400 Hz
t_ech2 = np.arange(0, 0.05, 1/fs2)
x_ech2 = A1*np.sin(2*np.pi*f1*t_ech2) + A2*np.sin(2*np.pi*f2*t_ech2) + A3*np.sin(2*np.pi*f3*t_ech2)

x_recons = reconstruction_shannon(x_ech2, fs2, t_ref)

plt.figure(figsize=(12, 4))
plt.plot(t_ref, x_ref, 'b-', label='Original', alpha=0.7)
plt.plot(t_ref, x_recons, 'r--', label='Reconstruit (fs=400 Hz)', alpha=0.7)
plt.stem(t_ech2, x_ech2, linefmt='g-', markerfmt='go', basefmt=' ', label='Échantillons')
plt.xlabel('Temps (s)')
plt.ylabel('Amplitude')
plt.title('Reconstruction à partir d\'échantillons sous-échantillonnés')
plt.legend()
plt.tight_layout()
plt.show()

# Erreur de reconstruction
erreur = np.mean((x_ref - x_recons)**2)
print(f"Erreur quadratique moyenne : {erreur:.4f}")
print("→ La reconstruction est incorrecte car le signal a été sous-échantillonné.")

### Interprétation B.3

- La reconstruction par **interpolation sinc** est **théoriquement exacte** si et seulement si $f_s > 2f_{\max}$.
- Ici, avec $f_s = 400$ Hz < 500 Hz, le signal reconstruit **diffère** du signal original.
- L'erreur quadratique moyenne est **non nulle** et significative.
- **Conclusion** : le sous-échantillonnage introduit des distorsions irréversibles. Il est **impératif** de respecter le théorème de Nyquist.

# Partie C – FFT et analyse spectrale (6 points)

## C.1 – Spectre d'amplitude et de phase

**Question C.1 (2 pts)** : Pour le signal $x(t)$ (partie A, sans bruit) :
1. Calculez la FFT.
2. Tracez le **spectre d'amplitude** (en dB) et le **spectre de phase**.
3. Identifiez les fréquences présentes.

In [ ]:
# --- Votre code ici ---
# Application d'une fenêtre de Hamming pour réduire les fuites
N = len(x)
window = np.hamming(N)
X = fft(x * window)
freqs = fftfreq(N, 1/fs)
freqs_pos = freqs[:N//2]
X_pos = X[:N//2]
mag = 2 * np.abs(X_pos) / np.sum(window)
mag_dB = 20 * np.log10(mag + 1e-12)
phase = np.angle(X_pos)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(freqs_pos, mag_dB)
axes[0].set_xlim(0, 400)
axes[0].set_ylim(-80, 5)
axes[0].set_xlabel('Fréquence (Hz)')
axes[0].set_ylabel('Magnitude (dB)')
axes[0].set_title('Spectre d\'amplitude (FFT + Hamming)')

axes[1].plot(freqs_pos, phase)
axes[1].set_xlim(0, 400)
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Phase (rad)')
axes[1].set_title('Spectre de phase')

plt.tight_layout()
plt.show()

# Identification automatique des pics
peaks, props = find_peaks(mag, height=0.05, distance=10)
print("Fréquences dominantes détectées :")
for idx in peaks:
    print(f"  {freqs_pos[idx]:.1f} Hz  (magnitude {mag[idx]:.3f})")

### Interprétation C.1

- Le spectre d'amplitude présente **3 pics nets** à **50, 120 et 250 Hz**, correspondant aux trois composantes du signal.
- Les amplitudes relatives (1.0, 0.6, 0.3) sont correctement retrouvées.
- Le **spectre de phase** est **bruité** pour les fréquences où l'amplitude est faible (bruit numérique). Aux fréquences des pics, la phase est **bien définie**.
- L'utilisation d'une **fenêtre de Hamming** atténue les fuites spectrales (lobes secondaires < −40 dB).

## C.2 – Effet du fenêtrage

**Question C.2 (2 pts)** : Comparez le spectre du signal **avec** et **sans** fenêtrage (rectangulaire vs Hamming vs Blackman).

1. Tracez les trois spectres superposés.
2. Observez l'effet sur les lobes secondaires et la largeur des pics.
3. Concluez sur le compromis résolution / dynamique.

In [ ]:
# --- Votre code ici ---
windows = {
    'Rectangulaire': np.ones(N),
    'Hamming': np.hamming(N),
    'Blackman': np.blackman(N)
}

plt.figure(figsize=(12, 5))
for name, w in windows.items():
    Xw = fft(x * w)
    mag_w = 2*np.abs(Xw[:N//2]) / np.sum(w)
    mag_w_dB = 20*np.log10(mag_w + 1e-12)
    plt.plot(freqs_pos, mag_w_dB, label=name, alpha=0.8)

plt.xlim(0, 400)
plt.ylim(-100, 5)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Comparaison des fenêtres (spectre d\'amplitude)')
plt.legend()
plt.tight_layout()
plt.show()

### Interprétation C.2

| Fenêtre | Largeur lobe principal | Niveau lobes secondaires | Compromis |
|---|---|---|---|
| **Rectangulaire** | Étroit (meilleure résolution) | Élevé (−13 dB) | Fuites importantes |
| **Hamming** | Moyen | Modéré (−43 dB) | Bon compromis |
| **Blackman** | Large | Faible (−58 dB) | Meilleure dynamique, résolution dégradée |

- La fenêtre **rectangulaire** offre la meilleure résolution mais génère des fuites spectrales qui peuvent masquer des composantes faibles.
- La fenêtre **de Hamming** est un bon compromis pour un usage général.
- La fenêtre **de Blackman** est recommandée lorsqu'on cherche à détecter des composantes de faible amplitude à côté de composantes fortes.

## C.3 – Zero-padding et résolution

**Question C.3 (2 pts)** :
1. Calculez la FFT du signal avec différents zero-paddings ($N_{fft} = N$, $2N$, $4N$).
2. Comparez la **résolution apparente** et la **résolution réelle**.
3. Concluez sur l'utilité du zero-padding.

In [ ]:
# --- Votre code ici ---
N_ffts = [N, 2*N, 4*N]
colors = ['C0', 'C1', 'C2']

plt.figure(figsize=(12, 5))
for N_fft, c in zip(N_ffts, colors):
    Xz = fft(x * np.hamming(N), N_fft)
    freqs_z = fftfreq(N_fft, 1/fs)[:N_fft//2]
    mag_z = 2*np.abs(Xz[:N_fft//2]) / np.sum(np.hamming(N))
    plt.plot(freqs_z, mag_z, color=c, label=f'N_fft = {N_fft}', alpha=0.8)

plt.xlim(40, 60)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude')
plt.title('Effet du zero-padding sur le pic à 50 Hz')
plt.legend()
plt.tight_layout()
plt.show()

print("→ Le zero-padding n'améliore PAS la résolution réelle (limitée par la durée d'observation).")
print("  Il permet seulement d'interpoler le spectre pour une meilleure visualisation.")

### Interprétation C.3

- La **résolution réelle** est fixée par la **durée d'observation** $T_{obs} = N/f_s$ :
$$\Delta f_{\text{réelle}} = \frac{f_s}{N} = \frac{1}{T_{obs}}$$
- Le **zero-padding** augmente le **nombre de points** de la FFT mais **ne crée pas d'information nouvelle**.
- Il **interpole** le spectre entre les points existants, ce qui donne une allure plus lisse mais **ne sépare pas** deux fréquences plus proches que $\Delta f_{\text{réelle}}$.
- **Utilité** : meilleure **visualisation**, estimation plus précise de la **position des pics** (interpolation), mais **pas** d'amélioration de la résolution.

# Partie D – Conception et application de filtres RIF/RII (5 points)

## D.1 – Filtre RIF (passe-bas)

**Question D.1 (2 pts)** : Concevez un filtre **RIF passe-bas** avec les spécifications suivantes :
- Fréquence de coupure : $f_c = 150$ Hz
- Fréquence d'échantillonnage : $f_s = 2000$ Hz
- Ordre : 64
- Méthode : fenêtre de Hamming

1. Calculez les coefficients du filtre.
2. Tracez la réponse impulsionnelle $h[n]$.
3. Tracez la réponse en fréquence $|H(f)|$ en dB.

In [ ]:
# --- Votre code ici ---
fc = 150  # Hz
ordre = 64

# Coefficients du filtre RIF avec fenêtre de Hamming
h_fir = signal.firwin(ordre + 1, fc, fs=fs, window='hamming')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Réponse impulsionnelle
axes[0].stem(np.arange(len(h_fir)), h_fir, basefmt=' ')
axes[0].set_xlabel('n')
axes[0].set_ylabel('h[n]')
axes[0].set_title('Réponse impulsionnelle du filtre RIF')

# Réponse en fréquence
w, H = freqz(h_fir, worN=2048, fs=fs)
axes[1].plot(w, 20*np.log10(np.abs(H) + 1e-12))
axes[1].axvline(fc, color='r', linestyle='--', label=f'fc = {fc} Hz')
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Magnitude (dB)')
axes[1].set_title('Réponse en fréquence')
axes[1].set_xlim(0, 500)
axes[1].set_ylim(-80, 5)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Nombre de coefficients : {len(h_fir)}")

### Interprétation D.1

- La **réponse impulsionnelle** est **symétrique** (filtre à phase linéaire).
- La réponse en fréquence présente une **bande passante** jusqu'à $f_c = 150$ Hz, puis une **bande atténuée** au-delà.
- Les **ondulations** dans la bande atténuée sont dues à la **troncature** de la réponse impulsionnelle idéale (phénomène de Gibbs).
- La fenêtre de Hamming permet de réduire ces ondulations (lobes secondaires < −40 dB).

## D.2 – Filtre RII (Butterworth)

**Question D.2 (2 pts)** : Concevez un filtre **RII passe-bas** de Butterworth avec les mêmes spécifications :
- $f_c = 150$ Hz, $f_s = 2000$ Hz
- Ordre 4

1. Calculez les coefficients $(b, a)$.
2. Tracez la réponse en fréquence.
3. Comparez avec le filtre RIF (nombre de coefficients, pente, phase).

In [ ]:
# --- Votre code ici ---
ordre_iiR = 4
b, a = butter(ordre_iiR, fc, btype='low', fs=fs)

w, H_iiR = freqz(b, a, worN=2048, fs=fs)

plt.figure(figsize=(12, 5))
plt.plot(w, 20*np.log10(np.abs(H) + 1e-12), label='RIF (Hamming, ordre 64)')
plt.plot(w, 20*np.log10(np.abs(H_iiR) + 1e-12), label='RII Butterworth (ordre 4)', linestyle='--')
plt.axvline(fc, color='r', linestyle=':', alpha=0.5, label=f'fc = {fc} Hz')
plt.xlim(0, 500)
plt.ylim(-80, 5)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Comparaison RIF vs RII')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Coefficients RIF : {len(h_fir)}")
print(f"Coefficients RII : b = {len(b)}, a = {len(a)}")

### Interprétation D.2

| Critère | RIF (Hamming) | RII (Butterworth) |
|---|---|---|
| **Nombre de coefficients** | 65 (ordre 64) | 5 (ordre 4) |
| **Phase** | Linéaire | Non linéaire |
| **Stabilité** | Toujours stable | Stable si pôles dans cercle unité |
| **Pente de transition** | Douce | Raide (pour un ordre faible) |
| **Ondulations** | Présentes (Gibbs) | Absentes (Butterworth) |

- Le filtre **RIF** nécessite beaucoup plus de coefficients pour une pente équivalente, mais garantit une **phase linéaire**.
- Le filtre **RII** est beaucoup plus **efficace** (ordre faible) mais introduit une **distorsion de phase**.
- **Choix** : RIF pour les applications audio (phase importante), RII pour les applications temps réel à ressources limitées.

## D.3 – Application du filtre

**Question D.3 (1 pt)** : Appliquez le filtre **RIF** et le filtre **RII** au signal bruité $x_{\text{bruité}}(t)$.

1. Tracez les signaux filtrés dans le domaine temporel.
2. Comparez les spectres avant/après filtrage.
3. Concluez sur l'efficacité du filtrage.

In [ ]:
# --- Votre code ici ---
# Filtrage RIF
y_fir = filtfilt(h_fir, [1.0], x_bruite)

# Filtrage RII
y_iiR = filtfilt(b, a, x_bruite)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(t[:200], x_bruite[:200], alpha=0.4, label='Bruité')
axes[0].plot(t[:200], y_fir[:200], label='Filtré RIF')
axes[0].plot(t[:200], y_iiR[:200], label='Filtré RII', linestyle='--')
axes[0].set_xlabel('Temps (s)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Signaux temporels')
axes[0].legend()

# Spectres
for sig_data, label, color in [(x_bruite, 'Bruité', 'C0'), (y_fir, 'Filtré RIF', 'C1'), (y_iiR, 'Filtré RII', 'C2')]:
    X_s = fft(sig_data * np.hamming(N))
    mag_s = 2*np.abs(X_s[:N//2]) / np.sum(np.hamming(N))
    axes[1].plot(freqs_pos, 20*np.log10(mag_s + 1e-12), label=label, alpha=0.8)

axes[1].axvline(fc, color='r', linestyle='--', alpha=0.5, label=f'fc = {fc} Hz')
axes[1].set_xlim(0, 400)
axes[1].set_ylim(-80, 5)
axes[1].set_xlabel('Fréquence (Hz)')
axes[1].set_ylabel('Magnitude (dB)')
axes[1].set_title('Spectres avant/après filtrage')
axes[1].legend()

plt.tight_layout()
plt.show()

print("→ Les composantes à 50 Hz et 120 Hz sont conservées, celle à 250 Hz est atténuée.")

### Interprétation D.3

- Les deux filtres **atténuent fortement** la composante à 250 Hz (au-delà de $f_c = 150$ Hz).
- Les composantes à **50 et 120 Hz** sont **conservées**.
- Le filtre **RIF** (Hamming, ordre 64) offre une meilleure **sélectivité** mais introduit un **retard de groupe** constant (compensé par `filtfilt`).
- Le filtre **RII** (Butterworth, ordre 4) est plus **rapide** (moins de coefficients) mais sa transition est plus **douce**.
- **Conclusion** : le filtrage fréquentiel permet d'isoler les composantes utiles d'un signal bruité.

# Synthèse de l'évaluation pratique – Partie 1

## Récapitulatif des compétences évaluées

| Compétence | Partie | Statut |
|---|---|---|
| Génération de signaux composés | A | ✅ |
| Ajout de bruit et calcul du SNR | A | ✅ |
| Théorème de Nyquist | B | ✅ |
| Simulation du repliement | B | ✅ |
| Reconstruction par interpolation sinc | B | ✅ |
| Calcul et interprétation de la FFT | C | ✅ |
| Comparaison des fenêtres | C | ✅ |
| Effet du zero-padding | C | ✅ |
| Conception de filtre RIF | D | ✅ |
| Conception de filtre RII | D | ✅ |
| Application et analyse du filtrage | D | ✅ |

## Points clés à retenir

1. **Échantillonnage** : $f_s > 2f_{\max}$ (Nyquist) – sinon repliement **irréversible**.
2. **Fenêtrage** : compromis **résolution / dynamique** – Hamming = bon compromis.
3. **Zero-padding** : interpole le spectre mais **n'améliore pas** la résolution réelle.
4. **RIF vs RII** : RIF = phase linéaire, ordre élevé ; RII = efficace, phase non linéaire.
5. **Filtrage** : permet d'isoler des composantes fréquentielles utiles.

## Préparation pour la Partie 2 (Semaine 13)

- Implémentation temps réel sur **STM32** avec **CMSIS-DSP**.
- Traitement de signaux **audio / ECG**.
- **Optimisation** des performances embarquées.

---

# Livrable Semaine 12

- Remplir **toutes** les cellules de code avec vos solutions.
- **Commenter** chaque résultat obtenu (interprétation physique).
- Les graphiques doivent être **lisibles** (titre, axes, légende, grille).
- Rendre le notebook complet (cellules exécutées).

**Checklist avant rendu** :
- [ ] Vérifier qu'aucune cellule ne produit d'erreur (`Kernel → Restart & Run All`)
- [ ] Toutes les **interprétations** sont présentes après chaque question
- [ ] Les **graphiques** sont lisibles (labels, légendes, titres)
- [ ] Le **SNR** est correctement calculé (en dB)
- [ ] Le **repliement** est clairement identifié et expliqué
- [ ] La **reconstruction** est comparée au signal original
- [ ] Les **fenêtres** sont comparées (résolution vs dynamique)
- [ ] Les **filtres RIF et RII** sont comparés (ordre, phase, pente)
- [ ] Exporter en PDF/HTML avec les sorties visibles

**Date de rendu** : à définir par l'enseignant.

---